**Notebook: MCP, Tools from the Sectors Server (Module 8)**
- OCBC AI Creator Masterclass, Day 2

___

This notebook is the hands-on part of Module 8 (about 32 minutes). After it come the example script
`day2/examples/m8_example.py` and the exercise `day2/exercises/m8_exercise.py`. Run it from the top, and run each
agent cell **once** (quota).

# Background

Your agent has four tools, and you wrote and maintain every one. The RM desk keeps asking for more: foreign flow,
most traded stocks, broker activity. Sectors already runs an **MCP server** with 66 ready-made tools. With **MCP**
your code connects to it, discovers the tools, keeps the few the job needs, and gives them to the agent next to your own:

- **8.1 Connect and discover.** The server connection and the client; the list of tools.
- **8.2 Call an MCP tool.** The server's own argument names; `await`.
- **8.3 Choose the tools.** From 66 to the few a job needs, by reading descriptions.
- **8.4 Mix MCP tools with your own.** One agent, tools from two places.

Who is the MCP **client**? **Our code**, not the LLM. The LLM still only asks for tool calls.

## Training Objectives

- Connect to the Sectors MCP server with Pydantic AI's `MCPToolset`
- Read an MCP tool's description and arguments, and call it with `await`
- Choose the MCP tools a job needs by searching their descriptions
- Build an agent with MCP tools and your own tools, and call it with `await agent.run(...)`

MCP tools are **asynchronous**: in a notebook you can write `await` directly in a cell.

## Checking your answers

**Your turn** cells say what the right result is. Run the next cell once for the **Knowledge Check** questions.

In [ ]:
import hashlib

ANSWERS = {'m8_client': 'ea7cbd4c6e', 'm8_worth': 'c38ff0a5de', 'm8_await': 'ba98753473', 'm8_all': '6e24c621f4', 'm8_rule': '0889f19e91'}

def check_answer(question, answer):
    key = hashlib.sha256(f"{question}:{answer.strip().lower()}".encode()).hexdigest()[:10]
    if answer.strip() == "":
        print("Type a, b, c or d between the quotes, then run the cell again.")
    elif key == ANSWERS[question]:
        print("Correct.")
    else:
        print("Not quite. Read the question again and try another answer.")

In [ ]:
import os
from dotenv import load_dotenv

# Work from the project root (the folder with data/), so file paths are the same as in the .py scripts
while not os.path.exists("data/sample"):
    os.chdir("..")

load_dotenv()  # reads the keys from .env
print("Working folder:", os.getcwd())

# 8.1 Connect and Discover

> **The request**
>
> Your manager: **"Before we write another tool ourselves, see what Sectors already offers."**

> **In this section**
>
> - **Key question:** How does our code connect to an MCP server, and what does it get back?
> - **How we run it:** **Live, about 8 minutes.**
> - **What you will do:** open a connection, list the tools, and look at their names.
> - **What carries forward:** `sectors` and `tools`, used in every section below.

| Piece | Here |
|---|---|
| **Server connection**: how to reach the server | `MCPToolset(SECTORS_MCP_URL, headers={"Authorization": <Sectors key>})` from `pydantic_ai.mcp` |
| **Client**: our side of the connection | the same `MCPToolset`: Pydantic AI builds it on the FastMCP client |
| **Toolset**: MCP tools → agent tools | pass it to `Agent(model, toolsets=[...])`, no adapter needed |

`async with` opens the session and closes it at the end of the block.

In [ ]:
from pydantic_ai.mcp import MCPToolset

sectors = MCPToolset(
    os.getenv("SECTORS_MCP_URL", "https://sectors-mcp.supertype.ai/mcp"),
    headers={"Authorization": os.getenv("SECTORS_API_KEY")},
)

async with sectors:
    tools = await sectors.list_tools()

print(len(tools), "tools on the Sectors MCP server")
print([t.name for t in tools][:12])

#### Knowledge Check: Who is the client?

In our system, which part is the MCP **client**?

- **a.** The OpenAI model
- **b.** Our Python code (Pydantic AI's `MCPToolset`)
- **c.** The Sectors server
- **d.** The RM's browser

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m8_client", answer)

#### Knowledge Check: When is MCP worth it?

When is connecting to an MCP server better than writing our own tool function around the REST API?

- **a.** Always
- **b.** Never
- **c.** When a provider already maintains many tools we want to reuse
- **d.** Only for Singapore data

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m8_worth", answer)

# 8.2 Call an MCP Tool

> **The request**
>
> An RM: **"How did BBCA trade in the week of 14 to 23 October?"** Use the server's tool, not ours.

> **In this section**
>
> - **Key question:** What does an MCP tool look like, and how do we call it ourselves?
> - **How we run it:** **Live, about 7 minutes.**
> - **What you will do:** read a tool's description and arguments, call it with `await`, then call another one yourself.
> - **What carries forward:** the habit of reading `.input_schema` before using a server's tool.

An MCP tool is like ours: a **name**, a **description** and **arguments** (in `.input_schema`), but it runs on the Sectors
server. Its arguments are the **server's** names: `symbol` instead of our `ticker`, and `sections` as a **list**.
We call it through the toolset: `await sectors.direct_call_tool(name, {...})`.

In [ ]:
async with sectors:
    tools = await sectors.list_tools()
    report_tool = next(t for t in tools if t.name == "fetch-company-report")
    print(report_tool.description[:250], "...")
    print(list(report_tool.input_schema["properties"]))
    nisp = await sectors.direct_call_tool("fetch-company-report", {"symbol": "NISP", "sections": ["overview"]})

print(str(nisp)[:300])

#### Your turn: Call fetch-daily-price

Find the tool `fetch-daily-price`, print its argument names, then call it for **BBCA** from `2026-10-14` to `2026-10-23`.
Print the first 300 characters of the result. The right result contains the closes of BBCA for those dates, ending with
`5025` on `2026-10-23`.

In [ ]:
## Your code below



#### Knowledge Check: Why `await`?

Why is the agent in this module called with `await agent.run(...)` instead of `agent.run_sync(...)`?

- **a.** `run` is cheaper
- **b.** MCP tools are asynchronous: they wait on the network, so the agent runs asynchronously
- **c.** The OpenAI model requires it
- **d.** It runs the tools in parallel on OpenAI's servers

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m8_await", answer)

# 8.3 Choose the Tools

> **The request**
>
> Your manager: **"The RM desk wants foreign flow and the most traded stocks. Which of the 66 tools do we need?"**

> **In this section**
>
> - **Key question:** How many MCP tools should an agent get, and how do we find the right ones?
> - **How we run it:** **Live, about 6 minutes.**
> - **What you will do:** measure what all 66 descriptions cost, keep a few by name, then find tools for a new job by searching descriptions.
> - **What carries forward:** `SELECTED`, the tools the agent gets in 8.4.

Every tool description is sent with **every** model call. Compare all 66 with the three our agent needs:

In [ ]:
SELECTED = ["fetch-company-report", "fetch-daily-price", "fetch-companies-top-changes"]

all_chars = sum(len(t.description) for t in tools)
selected_chars = sum(len(t.description) for t in tools if t.name in SELECTED)
print(f"All {len(tools)} descriptions: {all_chars:,} characters. The {len(SELECTED)} we keep: {selected_chars:,}")

#### Your turn: Find the tools for a new job

Without reading all 66 by eye, search the **descriptions** (lower case; leave out tools whose name contains `sgx` or
`klse`, which are other markets). First search for `"foreign"` and print the matches with the first line of each description:
you will get several tools, because brokers and shareholders can be foreign too. Then narrow it: search for `"foreign flow"`
and for `"most traded"`, and keep those names in `TRADING_TOOLS`. The right result is `['fetch-foreign-flow',
'fetch-most-traded-stocks']`: reading the descriptions, not guessing from names, is how you choose.

In [ ]:
## Your code below



#### Knowledge Check: Why not all 66?

Why not give the agent all 66 Sectors tools?

- **a.** MCP allows only five
- **b.** Every description goes into every prompt: more cost, and the LLM picks the wrong tool more often
- **c.** The OpenAI model refuses more than ten
- **d.** The other tools are not free

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m8_all", answer)

# 8.4 Mix MCP Tools with Your Own

> **The request**
>
> An RM: **"How did BMRI trade last week, and what does the news say about why?"** Prices from Sectors' server,
> reasons from **our** saved news.

> **In this section**
>
> - **Key question:** How does one agent use tools from the server and from our program?
> - **How we run it:** **Live, about 10 minutes.**
> - **What you will do:** add the Module 6 `search_news` tool to the MCP tools, run the agent with `await`, then build an agent for the new job.
> - **What carries forward:** the mixed agent that the example and the exercise build.

The agent sees one list of tools; where each runs does not matter to the model. **Rule of thumb:** MCP for the
provider's data, our own tools for what only we have (our saved news, our actions). The `search_news` tool is the same
as in Module 6, on the Module 5 news store. One new line in the system prompt: **today's date**. Our Module 6 tools computed dates in code; the MCP
tools take `start` and `end`, and the model does not know what day it is.

In [ ]:
import json
import numpy as np
from datetime import date
from pydantic_ai import Agent, Embedder
from pydantic_ai.messages import ToolCallPart
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.providers.openai import OpenAIProvider

model = OpenAIResponsesModel(
    os.getenv("OPENAI_MODEL", "gpt-6.1-sol"),
    provider=OpenAIProvider(api_key=os.getenv("OPENAI_API_KEY"), base_url=os.getenv("OPENAI_URL")),
)
embedder = Embedder("sentence-transformers:all-mpnet-base-v2")   # open source, runs on your laptop

# The Module 5 news store: split, embed, search
def split_text(text, chunk_size=1500, chunk_overlap=100):
    """Cut a text into pieces of chunk_size characters; each piece repeats the last chunk_overlap characters of the one before."""
    chunks, start = [], 0
    while True:
        chunks.append(text[start:start + chunk_size])
        if start + chunk_size >= len(text):
            return chunks
        start += chunk_size - chunk_overlap


def split_documents(docs, chunk_size=1500, chunk_overlap=100):
    return [{"text": piece, "metadata": doc["metadata"]}
            for doc in docs for piece in split_text(doc["text"], chunk_size, chunk_overlap)]


def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


async def build_vector_store(chunks):
    vectors = (await embedder.embed_documents([c["text"] for c in chunks])).embeddings
    return {"chunks": chunks, "vectors": np.array(vectors)}


async def similarity_search(vector_store, query, k=4):
    query_vector = (await embedder.embed_query(query))[0]
    scored = [(cosine_similarity(query_vector, v), c) for v, c in zip(vector_store["vectors"], vector_store["chunks"])]
    return [c for s, c in sorted(scored, key=lambda pair: pair[0], reverse=True)[:k]]


docs, seen = [], set()
for ticker in ["BBCA", "BBRI", "BMRI", "NISP"]:
    with open(f"data/sample/news_{ticker}.json", encoding="utf-8") as f:
        for article in json.load(f):
            if article["title"] not in seen:
                seen.add(article["title"])
                docs.append({"text": article["title"] + "\n\n" + article["body"],
                             "metadata": {"title": article["title"], "date": article["timestamp"][:10]}})
vector_store = await build_vector_store(split_documents(docs))


async def search_news(query: str) -> str:
    """Search recent Indonesian market news. Use it to find WHY something happened
    (events, announcements, reasons). It does not give prices.

    Args:
        query: what to look for, in plain words.
    """
    results = await similarity_search(vector_store, query, k=4)
    return "\n\n".join(f"{c['metadata']['title']} ({c['metadata']['date']})\n{c['text']}" for c in results)


SYSTEM_PROMPT = f"""You are the IDX Market Assistant for relationship managers at a bank in Indonesia.
Today is {date.today()}. The MCP tools take explicit dates: work them out from today.
Use the tools to get data; never guess a number. Say the date of every price you quote.
Use search_news to explain why something happened. Never give buy, sell or hold advice."""


def tools_used(result):
    return [p.tool_name for m in result.all_messages() for p in m.parts if isinstance(p, ToolCallPart)]


async with sectors:
    agent = Agent(model, toolsets=[sectors.filtered(lambda ctx, tool_def: tool_def.name in SELECTED)],
                  tools=[search_news], instructions=SYSTEM_PROMPT)
    result = await agent.run("How did BMRI trade over the last week, and what does the news say about why?")

print("Tools used:", tools_used(result))
print(result.output)

#### Your turn: An agent for the trading desk

Build an agent with your `TRADING_TOOLS` (from 8.3) plus `search_news`, inside one `async with` block, and ask
*"Was BBRI bought or sold by foreign investors over the last two weeks, and what does the news say about it?"* Print the tools
used and the answer. The right result uses `fetch-foreign-flow` with `symbol` `BBRI` **and** `search_news`, and the answer
says whether the net foreign flow was positive (buying) or negative (selling), with dates.

In [ ]:
## Your code below



#### Knowledge Check: MCP or our own tool?

The RMs want the agent to search **OCBC's internal research notes**. Where should that tool come from?

- **a.** The Sectors MCP server
- **b.** Our own tool function, in our program: it is our data, not the provider's
- **c.** The OpenAI model's built-in search
- **d.** It cannot be a tool

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m8_rule", answer)

# Module 8 Wrap-up

| Idea | In code |
|---|---|
| Connect | `MCPToolset(url, headers=...)` from `pydantic_ai.mcp`, inside `async with` |
| Discover | `await sectors.list_tools()`: 66 MCP tools |
| Call | `await sectors.direct_call_tool(name, {...})` with the server's argument names |
| Choose | keep a few by name with `sectors.filtered(...)`; search descriptions to find them |
| Mix | `Agent(model, toolsets=[mcp_tools], tools=[search_news])`, `await agent.run(...)` |

**Next:** `day2/examples/m8_example.py`, then the exercise `day2/exercises/m8_exercise.py`. In Module 9 the scheduler
from Day 1 wakes this agent up.